# Variance97 — Phase 4: Data Pipeline
## Self-updating dataset from the NHL API

This notebook is the user-facing demo of the Phase 4 pipeline. The *implementation* lives in `data/build/` as plain Python modules — that's where it should live, since the pipeline is meant to run on a cron / CI schedule, not from a notebook. This notebook walks through what each step does and verifies the outputs.

### What the pipeline does

1. For each tracked player (McDavid, MacKinnon), find the latest game date in their NHL source CSV.
2. Fetch any newer games from `api-web.nhle.com`.
3. Boxscore-enrich new rows with `result`, `team_score`, `opp_score` (the gameLog endpoint omits these).
4. Append new rows to the player's NHL source CSV (idempotent — duplicates dropped on date).
5. Refresh `opponent_team_stats.csv` from current standings.
6. Run `apply_features.py` to recompute `is_elimination_game` (rule-based), `rolling_pts_5`, `rest_days`, `is_back_to_back`, and `opp_ga_per_game` from scratch on the merged dataset (NHL source + manual international entries).
7. Write the merged + featured `*_game_log_clean.csv` files that the analysis notebooks consume.

### Module layout

```
data/
  build/
    fetch_player_log.py    # generic NHL gameLog fetcher (parameterized by player)
    fetch_boxscores.py     # adds result/team_score/opp_score per game
    fetch_team_stats.py    # NHL standings -> per-team GA/game by season
    apply_features.py      # is_elimination_game + ML features
    update_all.py          # orchestrator (cron/CI entry point)
  international_games.csv  # manual entry for Four Nations / Olympics
  mcdavid_nhl_log.csv      # API-derived NHL source (rebuilt by pipeline)
  mackinnon_nhl_log.csv    # API-derived NHL source (rebuilt by pipeline)
  opponent_team_stats.csv  # per-season team GA/game (rebuilt by pipeline)
  mcdavid_game_log_clean.csv     # final merged + featured (analysis input)
  mackinnon_game_log_clean.csv   # final merged + featured (analysis input)
```

## 1. Pre-state — what's currently in the dataset

Before running the pipeline, inspect the existing source CSVs to confirm we know the cursor (latest game date) for each player.

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path('../data')

for name, path in {
    'McDavid NHL source': DATA / 'mcdavid_nhl_log.csv',
    'MacKinnon NHL source': DATA / 'mackinnon_nhl_log.csv',
    'International (manual)': DATA / 'international_games.csv',
    'Team stats': DATA / 'opponent_team_stats.csv',
}.items():
    if path.exists():
        df = pd.read_csv(path)
        latest = df['date'].max() if 'date' in df.columns else 'n/a'
        print(f"{name:25s}  rows={len(df):>4d}  latest={latest}")
    else:
        print(f"{name:25s}  (missing)")

McDavid NHL source         rows= 468  latest=2026-04-30
MacKinnon NHL source       rows= 428  latest=2026-05-05
International (manual)     rows=  10  latest=2026-02-22
Team stats                 rows= 160  latest=n/a


## 2. Run the pipeline

Calls `data.build.update_all.main()` directly. Equivalent to running `python -m data.build.update_all` from the repo root. Idempotent: runs with no new games will report `+0` and exit cleanly.

In [2]:
import sys
sys.path.insert(0, str(DATA / 'build'))

from update_all import main as run_pipeline
run_pipeline()

=== variance97 update pipeline ===
current season window: 2025-26

[Connor McDavid] refreshing mcdavid_nhl_log.csv
  existing rows: 468  (latest: 2026-04-30)


  Connor McDavid 20212022 type=2: 80 games


  Connor McDavid 20212022 type=3: 16 games


  Connor McDavid 20222023 type=2: 82 games


  Connor McDavid 20222023 type=3: 12 games


...

  2021-22 (2022-04-30): 32 teams


  2022-23 (2023-04-13): 32 teams


  2023-24 (2024-04-18): 32 teams


  2024-25 (2025-04-15): 32 teams


  2025-26 (2026-04-15): 32 teams
  wrote 160 rows.

[features] applying for McDavid (with international concat)
Total rows: 478  (elimination games: 17)
Feature coverage:
  rest_days non-null:       477/478
  rolling_pts_5 non-null:   477/478
  opp_ga_per_game non-null: 449/478
Wrote -> /Users/Kylan/Desktop/Variance97/data/mcdavid_game_log_clean.csv

[features] applying for MacKinnon (NHL only)
Total rows: 428  (elimination games: 6)
Feature coverage:
  rest_days non-null:       427/428
  rolling_pts_5 non-null:   427/428
  opp_ga_per_game non-null: 428/428
Wrote -> /Users/Kylan/Desktop/Variance97/data/mackinnon_game_log_clean.csv

=== done. mcdavid +0, mackinnon +0 ===


## 3. Verify the merged + featured outputs

These are the files the analysis notebooks consume. After the pipeline runs, every derived column should be present and feature coverage should be near 100% (with the expected NaNs: first row has no `rest_days` / `rolling_pts_5`; international opponents have no `opp_ga_per_game`).

In [3]:
for name, path in {
    'McDavid clean': DATA / 'mcdavid_game_log_clean.csv',
    'MacKinnon clean': DATA / 'mackinnon_game_log_clean.csv',
}.items():
    df = pd.read_csv(path)
    derived = ['is_elimination_game', 'rest_days', 'is_back_to_back', 'rolling_pts_5', 'opp_ga_per_game']
    print(f"\n=== {name}  ({len(df)} rows) ===")
    print(f"Columns: {list(df.columns)}")
    for c in derived:
        if c in df.columns:
            present = df[c].notna().sum()
            print(f"  {c:22s}  non-null: {present}/{len(df)}")
    print(f"  date range: {df['date'].min()} -> {df['date'].max()}")


=== McDavid clean  (478 rows) ===
Columns: ['date', 'opponent', 'goals', 'assists', 'points', 'plus_minus', 'SOG', 'TOI', 'result', 'team_score', 'opp_score', 'game_number', 'game_context', 'season', 'is_elimination_game', 'rest_days', 'is_back_to_back', 'rolling_pts_5', 'opp_ga_per_game']
  is_elimination_game     non-null: 478/478
  rest_days               non-null: 477/478
  is_back_to_back         non-null: 478/478
  rolling_pts_5           non-null: 477/478
  opp_ga_per_game         non-null: 449/478
  date range: 2021-10-13 -> 2026-04-30

=== MacKinnon clean  (428 rows) ===
Columns: ['gameId', 'date', 'opponent', 'goals', 'assists', 'points', 'plus_minus', 'SOG', 'TOI', 'result', 'team_score', 'opp_score', 'game_number', 'game_context', 'season', 'is_elimination_game', 'home_away', 'rest_days', 'is_back_to_back', 'rolling_pts_5', 'opp_ga_per_game']
  is_elimination_game     non-null: 428/428
  rest_days               non-null: 427/428
  is_back_to_back         non-null: 428/428


In [4]:
# Sanity-check the rule-derived elimination games for McDavid.
mcdavid = pd.read_csv(DATA / 'mcdavid_game_log_clean.csv')
elim = mcdavid[mcdavid['is_elimination_game']]
print(f"Elimination games flagged: {len(elim)}")
print()
print(elim[['date', 'opponent', 'game_context', 'game_number', 'result', 'points']].to_string(index=False))

Elimination games flagged: 17

      date opponent                game_context  game_number result  points
2022-05-12      LAK                 first_round          6.0      W       3
2022-05-14      LAK                 first_round          7.0      W       2
2022-06-06      COL                 conf_finals          4.0      L       3
2023-05-14      VEG                second_round          6.0      L       1
2024-05-18      VAN                second_round          6.0      W       3
2024-05-20      VAN                second_round          7.0      W       0
2024-06-15      FLA          stanley_cup_finals          4.0      W       4
2024-06-18      FLA          stanley_cup_finals          5.0      W       4
2024-06-21      FLA          stanley_cup_finals          6.0      W       0
2024-06-24      FLA          stanley_cup_finals          7.0      L       0
2025-02-20      USA four_nations_faceoff_finals          4.0      W       1
2025-06-17      FLA          stanley_cup_finals          

## 4. International games — the manual workflow

The NHL API does not cover Four Nations Face-Off or the Winter Olympics. Those games are entered manually into `data/international_games.csv` (same schema as the NHL source CSV; derived columns are recomputed by the pipeline). The pipeline concatenates this file before applying features, so any manual additions automatically flow into the analysis-ready CSV.

To add a new international game:

1. Open `data/international_games.csv`.
2. Append a row with `date`, `opponent` (3-letter country code), `goals`, `assists`, `points`, `plus_minus`, `SOG`, `TOI` (decimal minutes), `result` (W/L), `team_score`, `opp_score`, `game_number`, `game_context` (one of `four_nations_faceoff_group`, `four_nations_faceoff_finals`, `olympics_exhibition`, `olympics_quarterfinals`, `olympics_semifinals`, `olympics_finals`), and `season`.
3. Re-run `update_all.main()` (or `python data/build/update_all.py`).

The current contents:

In [5]:
intl = pd.read_csv(DATA / 'international_games.csv')
print(f"International rows: {len(intl)}")
print(intl[['date', 'opponent', 'game_context', 'points', 'result']].to_string(index=False))

International rows: 10
      date opponent                game_context  points result
2025-02-12      SWE  four_nations_faceoff_group       1      W
2025-02-15      USA  four_nations_faceoff_group       1      L
2025-02-17      FIN  four_nations_faceoff_group       2      W
2025-02-20      USA four_nations_faceoff_finals       1      W
2026-02-12      CZH         olympics_exhibition       3      W
2026-02-13      SUI         olympics_exhibition       3      W
2026-02-15      FRA         olympics_exhibition       3      W
2026-02-18      CZH      olympics_quarterfinals       2      W
2026-02-20      FIN         olympics_semifinals       2      W
2026-02-22      USA             olympics_finals       0      L


## 5. Scheduling

The pipeline is designed to be safe to run on a cron. A daily run during the season is plenty — most days have at most one Edmonton + one Colorado game between them.

**Local cron** (macOS / Linux):

```bash
# daily at 4am local
0 4 * * *  cd /path/to/Variance97 && bash scripts/run_update.sh >> /tmp/variance97.log 2>&1
```

**GitHub Actions** (stretch, in `PHASE4_PLAN.md`): a scheduled workflow that runs `python -m data.build.update_all`, commits any CSV updates, and pushes back to `main`. Good fit because the dataset is the deployable artifact.

## What this phase intentionally does NOT include

- **Streamlit app** — Phase 5.
- **Goalie-specific features** (Limitation #5) — would require boxscore-level goalie stats and per-season save% data; meaningful follow-on engineering.
- **Additional peer players** (Limitation #3) — the `fetch_player_log.PLAYERS` registry makes adding Matthews / Crosby / Draisaitl a one-line change, but expanding the peer set is its own analytical decision.
- **Real-time updates** — daily is sufficient. Faster cadence is over-engineering.

See `LIMITATIONS.md` and `PHASE4_PLAN.md` at the repo root for the full list of what's in scope and what's deliberately out of scope.